# Quantum Prediction Visualization
Comparing predicted landmarks (from quantum model) vs actual landmarks (from MediaPipe)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

## Load Data

In [ ]:
# Load predicted landmarks
with open('../data/quant-gen-vids/test.json') as f:
    predicted_data = json.load(f)

# Load actual landmarks for video 11
with open('../data/landmarks/11_smooth-hand-movment.json') as f:
    actual_data_11 = json.load(f)

# Load actual landmarks for video 12
with open('../data/landmarks/12_typing.json') as f:
    actual_data_12 = json.load(f)

print(f'Predicted frames: {len(predicted_data)}')
print(f'Actual frames (video 11): {len(actual_data_11)}')
print(f'Actual frames (video 12): {len(actual_data_12)}')

## Match Predictions with Actual by Timestamp

In [ ]:
# Build lookup for actual landmarks by (video, timestamp)
actual_lookup = {}
for entry in actual_data_11:
    actual_lookup[('11_smooth-hand-movment', entry['timestamp'])] = entry['landmarks']
for entry in actual_data_12:
    actual_lookup[('12_typing', entry['timestamp'])] = entry['landmarks']

# Match predicted with actual
from pathlib import Path
matched = []
for pred in predicted_data:
    video_name = Path(pred['videopath']).stem
    ts = pred['timestamp_ms']
    key = (video_name, ts)
    if key in actual_lookup:
        matched.append({
            'predicted': pred['landmarks'],
            'actual': actual_lookup[key],
            'timestamp': ts,
            'video': video_name
        })

print(f'Matched frames: {len(matched)}')
print(f'Sample video: {matched[0]["video"] if matched else "No matches"}')

## Plot: Predicted vs Actual Hand (Single Frame)

In [ ]:
# MediaPipe hand connections for drawing lines between joints
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),       # Thumb
    (0,5),(5,6),(6,7),(7,8),       # Index
    (0,9),(9,10),(10,11),(11,12),  # Middle
    (0,13),(13,14),(14,15),(15,16),# Ring
    (0,17),(17,18),(18,19),(19,20),# Pinky
    (5,9),(9,13),(13,17)           # Palm
]

def plot_hand(ax, landmarks, title, color='blue'):
    """Plot a hand skeleton from 21 landmarks"""
    lm = np.array(landmarks)
    ax.scatter(lm[:, 0], lm[:, 1], c=color, s=30, zorder=5)
    for (i, j) in HAND_CONNECTIONS:
        ax.plot([lm[i,0], lm[j,0]], [lm[i,1], lm[j,1]], c=color, alpha=0.6)
    ax.set_title(title)
    ax.invert_yaxis()  # MediaPipe y-axis is inverted
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

In [ ]:
# Pick a frame to compare
frame_idx = 0  # Change this to view different frames
sample = matched[frame_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_hand(ax1, sample['actual'], f"Actual (t={sample['timestamp']}ms)", color='green')
plot_hand(ax2, sample['predicted'], f"Predicted (t={sample['timestamp']}ms)", color='red')
fig.suptitle(f"Video: {sample['video']}", fontsize=14)
plt.tight_layout()
plt.show()

## Overlay: Both Hands on Same Plot

In [ ]:
frame_idx = 0  # Change this to view different frames
sample = matched[frame_idx]

fig, ax = plt.subplots(figsize=(8, 8))
plot_hand(ax, sample['actual'], '', color='green')
plot_hand(ax, sample['predicted'], '', color='red')
ax.set_title(f"Overlay — Green=Actual, Red=Predicted (t={sample['timestamp']}ms)")
plt.tight_layout()
plt.show()

## Error Analysis

In [ ]:
# Calculate per-joint error across all matched frames
all_errors = []
for m in matched:
    pred = np.array(m['predicted'])
    actual = np.array(m['actual'])
    # Euclidean distance per joint
    errors = np.sqrt(np.sum((pred - actual)**2, axis=1))
    all_errors.append(errors)

all_errors = np.array(all_errors)
mean_errors = all_errors.mean(axis=0)

joint_names = ['Wrist','Thumb_CMC','Thumb_MCP','Thumb_IP','Thumb_Tip',
               'Index_MCP','Index_PIP','Index_DIP','Index_Tip',
               'Middle_MCP','Middle_PIP','Middle_DIP','Middle_Tip',
               'Ring_MCP','Ring_PIP','Ring_DIP','Ring_Tip',
               'Pinky_MCP','Pinky_PIP','Pinky_DIP','Pinky_Tip']

print(f'Overall MSE: {(all_errors**2).mean():.4f}')
print(f'Overall Mean Error: {all_errors.mean():.4f}')
print(f'\nPer-joint mean error:')
for name, err in zip(joint_names, mean_errors):
    print(f'  {name:15s}: {err:.4f}')

In [ ]:
# Bar chart of per-joint errors
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(21), mean_errors, color='coral')
ax.set_xticks(range(21))
ax.set_xticklabels(joint_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Euclidean Error')
ax.set_title('Prediction Error by Joint')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Trajectory Over Time (Single Joint)

In [ ]:
# Plot x-coordinate of wrist (joint 0) over time
# Filter to video 11 only
v11 = [m for m in matched if '11' in m['video']]
timestamps = [m['timestamp'] for m in v11]
pred_x = [m['predicted'][0][0] for m in v11]   # joint 0, x coord
actual_x = [m['actual'][0][0] for m in v11]     # joint 0, x coord

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(timestamps, actual_x, 'g-', label='Actual', alpha=0.8)
ax.plot(timestamps, pred_x, 'r--', label='Predicted', alpha=0.8)
ax.set_xlabel('Timestamp (ms)')
ax.set_ylabel('X coordinate')
ax.set_title('Wrist X-Position Over Time (Video 11)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()